In [1]:
import sys
sys.path.append("..")

import pandas as pd
from pathlib import Path
from src.clean import clean_matches, clean_deliveries
from src.features import (
    build_batting_features,
    build_bowling_features,
    build_match_summary,
    build_team_season_stats,
)
RAW = Path(r"E:\ipl-analytics\data\raw")
PROCESSED = Path(r"E:\ipl-analytics\data\processed")

In [2]:
matches_raw = pd.read_csv(RAW / "matches.csv")
deliveries_raw = pd.read_csv(RAW / "deliveries.csv", low_memory=False)

print(f"RAW matches: {matches_raw.shape}")
print(f"RAW deliveries: {matches_raw.shape}")
print(f"\nRaw season (before cleaning):")
print(sorted(matches_raw["season"].astype(str).unique()))

RAW matches: (1209, 20)
RAW deliveries: (1209, 20)

Raw season (before cleaning):
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']


In [3]:
matches = clean_matches(matches_raw)
deliveries = clean_deliveries(deliveries_raw)

OK — all seasons are valid IPL years.
OK — all expected IPL seasons are present.

Season distribution : [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Season range        : 2008 – 2026
Total seasons       : 19
Total matches       : 1209

clean_matches done: (1209, 20) → (1209, 24)
Remaining nulls:
city              51
match_type       114
target_overs     117
super_over       114
method          1188
umpire1          114
umpire2          114

clean_deliveries: (287513, 17) → (287513, 24)
  Nulls remaining:
extras_type    273388
fielder        277108


In [4]:
assert pd.api.types.is_datetime64_any_dtype(matches["date"]), \
   "date column should be datetime64"

matches["season"] = pd.to_numeric(matches["season"], errors='coerce').astype('int64')

assert matches["season"].dtype == int or matches["season"].dtype == "int64", \
    "season should be int"

assert matches["season"].min() == 2008, \
    f"Earliest season should be 2008, got {matches['season'].min()}"

assert 2007 not in matches["season"].values, \
    "2007 should not exist - IPL started in 2008"

assert deliveries["is_wicket"].sum() > 0, \
    "is_ wicket column looks wrong - no wickets found"

assert deliveries["is_four"].sum() > 0, \
    "is_four columns wrong - no fours found"

assert (
    deliveries["match_id"].isin(matches["match_id"]).all()
), "some delivery match_ids have no corresponding match row"   

print("All assertions passed.")

All assertions passed.


In [5]:
print("Raw season values that might be 2020:")
raw_seasons = matches_raw["season"].astype(str).unique()
candidates  = [s for s in raw_seasons if "20" in s]
print(sorted(candidates))

if "date" in matches_raw.columns:
    matches_raw["_date"] = pd.to_datetime(matches_raw["date"], errors="coerce")
    year_2020 = matches_raw[matches_raw["_date"].dt.year == 2020]
    print(f"\nMatches with dates in 2020: {len(year_2020)}")
    if len(year_2020) > 0:
        print(f"Their season values: {year_2020['season'].unique()}")

Raw season values that might be 2020:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Matches with dates in 2020: 60
Their season values: <StringArray>
['2020/21']
Length: 1, dtype: str


In [6]:
batting_stats = build_batting_features(deliveries)
bowling_stats = build_bowling_features(deliveries)
match_summary = build_match_summary(matches, deliveries)
team_season_stats = build_team_season_stats(match_summary)

build_batting_features: 460 batters with 30+ balls faced
build_bowling_features: 402 bowlers with 60+ balls bowled
build_match_summary: (1209, 45)
build_team_season_stats: (166, 7)


In [7]:
matches.to_csv(PROCESSED / "matches_clean.csv", index=False)
deliveries.to_csv(PROCESSED / "deliveries_clean.csv", index=False)
batting_stats.to_csv(PROCESSED / "batting_features.csv", index=False)
bowling_stats.to_csv(PROCESSED / "bowling_features.csv", index=False)
match_summary.to_csv(PROCESSED / "match_summary.csv", index=False)
team_season_stats.to_csv(PROCESSED / "team_season_stats.csv", index=False)

print("\nAll files saved to data/processed/")


All files saved to data/processed/


In [8]:
import sqlite3

conn = sqlite3.connect(PROCESSED / "ipl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)
deliveries.to_sql("deliveries", conn, if_exists="replace", index=False)
batting_stats.to_sql("batting", conn, if_exists="replace", index=False)
bowling_stats.to_sql("bowling", conn, if_exists="replace", index=False)
match_summary.to_sql("match_summary", conn, if_exists="replace", index=False)
team_season_stats.to_sql("team_season", conn, if_exists="replace", index=False)
conn.close()

print("SQLite DB saved: data/processed/ipl.db")

SQLite DB saved: data/processed/ipl.db


In [9]:
print("\n=== OUTPUT SUMMARY ===")
for name, df in {
    "matches_clean" :  matches,
    "deliveries_clean": deliveries,
    "batting_features": batting_stats,
    "bowling_features": bowling_stats,
    "match_summary": match_summary,
    "team_season_stats": team_season_stats,
}.items():
    nulls = df.isnull().sum().sum()
    print(f" {name:<22}: shape={str(df.shape):<16} nulls={nulls}")


=== OUTPUT SUMMARY ===
 matches_clean         : shape=(1209, 24)       nulls=1812
 deliveries_clean      : shape=(287513, 24)     nulls=550496
 batting_features      : shape=(460, 19)        nulls=164
 bowling_features      : shape=(402, 20)        nulls=37
 match_summary         : shape=(1209, 45)       nulls=1866
 team_season_stats     : shape=(166, 7)         nulls=0


In [10]:
print("\n=== Top 10 run scorers ===")
print(batting_stats[["batter", "total_runs", "batting_average", "strike_rate", "hundreds", "fifties"]].head(10).to_string(index=False))

print("\n=== Top 10 wicket takers ===")
print(bowling_stats[["bowler", "wickets", "economy_rate", "bowling_average", "dot_ball_pct"]].head(10).to_string(index=False))

print("\n=== New columns in deliveries ===")
new_cols = ["is_wicket", "is_four", "is_six", "is_dot_ball", "over_phase", "is_legal_delivery"]
print(deliveries[new_cols].head(10).to_string(index=False))


=== Top 10 run scorers ===
        batter  total_runs  batting_average  strike_rate  hundreds  fifties
       V Kohli        9022            38.07       133.80         8       66
     RG Sharma        7185            28.86       132.64         2       48
      S Dhawan        6769            35.07       127.62         2       51
     DA Warner        6567            40.04       140.26         4       62
      KL Rahul        5593            45.47       138.17         6       42
      SK Raina        5536            33.15       137.54         1       39
      MS Dhoni        5439            34.42       137.91         0       24
     AM Rahane        5194            29.85       124.92         2       34
AB de Villiers        5181            41.45       152.38         3       40
     SV Samson        5008            31.90       140.83         5       26

=== Top 10 wicket takers ===
    bowler  wickets  economy_rate  bowling_average  dot_ball_pct
 YS Chahal      237          8.11        

In [11]:
# Quick check — run this in a notebook
import pandas as pd
summary = pd.read_csv(r"E:/ipl-analytics/data/processed/match_summary.csv")
print("batting_first_won" in summary.columns)
print(summary.columns.tolist())

True
['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2', 'result_type', 'is_no_result', 'day_of_week', 'month', 'inn1_batting_team', 'inn1_total_runs', 'inn1_total_wickets', 'inn1_total_balls', 'inn1_total_fours', 'inn1_total_sixes', 'inn1_dot_balls', 'inn1_run_rate', 'inn2_batting_team', 'inn2_total_runs', 'inn2_total_wickets', 'inn2_total_balls', 'inn2_total_fours', 'inn2_total_sixes', 'inn2_dot_balls', 'inn2_run_rate', 'toss_winner_won', 'batting_first_team', 'batting_first_won', 'score_diff', 'season_num']


In [12]:
# In callbacks.py, check these lines at the top
matches    = pd.read_csv(PROCESSED / "matches_clean.csv",  parse_dates=["date"])
print(matches.columns.tolist()[:5])

['match_id', 'season', 'city', 'date', 'match_type']


In [13]:
import json
with open(r"E:/ipl-analytics/data/processed/models/win_prob_features.json") as f:
    print(json.load(f))
    

['A_form5', 'B_form5', 'form_diff5', 'A_form10', 'B_form10', 'form_diff10', 'A_overall_wr', 'B_overall_wr', 'overall_wr_diff', 'A_venue_wr', 'B_venue_wr', 'venue_wr_diff', 'venue_bat_first_wr', 'A_avg_score', 'B_avg_score', 'score_diff', 'A_avg_wickets', 'B_avg_wickets', 'wicket_diff', 'h2h_wr_A', 'h2h_n', 'toss_decision_bat', 'toss_correct', 'A_consistency', 'B_consistency', 'season_stage']


In [14]:
import pandas as pd
df = pd.read_csv(r"E:/ipl-analytics/data/processed/match_summary.csv")
print("batting_first_won" in df.columns)
print(df.shape)

True
(1209, 45)


In [17]:
# In your notebook or terminal
import pandas as pd
deliveries = pd.read_csv(r"E:/ipl-analytics/data/processed/deliveries_clean.csv")

# Check what batting_team values look like
print(deliveries["batting_team"].value_counts().head(5))
print(deliveries.columns.tolist())

C:\Users\BHUMI\AppData\Local\Temp\ipykernel_25556\3219913361.py:3: DtypeWarning: Columns (0: extras_type) have mixed types. Specify dtype option on import or set low_memory=False.
  deliveries = pd.read_csv(r"E:/ipl-analytics/data/processed/deliveries_clean.csv")


batting_team
Mumbai Indians                 34133
Royal Challengers Bangalore    32592
Delhi Capitals                 32365
Kolkata Knight Riders          31779
Chennai Super Kings            31414
Name: count, dtype: int64
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'is_wide', 'is_noball', 'is_four', 'is_six', 'is_dot_ball', 'over_phase', 'is_legal_delivery']
